# Image Generation Fundamentals

**Module:** 17 — Image Generation

Text-to-image systems from product surface to training, inference, alignment, evaluation, and safety.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain text-to-image systems at a product and ML level
- Distinguish training, inference, and alignment stages
- Map conditioning signals to txt2img, img2img, inpaint, outpaint
- Identify evaluation criteria beyond aesthetics
- Apply a safety and rights checklist to a generation feature


## The Job of an Image Generator

### Definition
An image generator maps a **conditioning signal** (text, reference image, depth map, mask, style embedding) to pixels that satisfy semantic, aesthetic, and policy constraints.

### Why it matters
Products sell controllable visual assets. Without a clear conditioning→pixel contract, quality and safety become un-debuggable vibes.

### How it works
Encode the condition → sample in latent or pixel space → decode → safety/post filters → return bytes + metadata (seed, model, credentials).

### Intuition
A skilled illustrator plus a studio pipeline that rejects unsafe or brand-breaking work.

### Pitfalls
- No seed/params logging
- Optimizing only aesthetics
- Shipping without content credentials

### When to use
Any feature that turns language or control signals into images at scale.


### Product surface & quality axes

| Mode | Input | Use |
|------|-------|-----|
| **txt2img** | Prompt (+ negative) | Ideation, stock-like assets |
| **img2img** | Image + prompt + strength | Restyle, variation |
| **Inpaint** | Image + mask + prompt | Object replace/remove |
| **Outpaint** | Image + expand bounds | Aspect / canvas extend |
| **Variations** | Seed neighborhood | Explore nearby samples |

```mermaid
flowchart LR
  C[Conditioning] --> E[Encode]
  E --> S[Sample / Denoise]
  S --> D[Decode]
  D --> F[Safety + Post]
  F --> O[Image + Metadata]
```

| Axis | Question |
|------|----------|
| Prompt adherence | Did requested subjects appear? |
| Aesthetics | Composition, lighting, style |
| Text / logos | Legible when needed? |
| Anatomy | Hands, faces, perspective? |
| Diversity | Seed sweep yields distinct options? |
| Latency / cost | Meets SLA and unit economics? |
| Safety | Blocks disallowed content / likeness? |


In [ ]:
# Demo 1: generation job + weighted rubric
from dataclasses import dataclass, field
from typing import Any
import hashlib

@dataclass
class ImageJob:
    prompt: str
    negative: str = ""
    width: int = 1024
    height: int = 1024
    steps: int = 28
    cfg: float = 7.0
    seed: int = 42
    mode: str = "txt2img"
    meta: dict[str, Any] = field(default_factory=dict)

    @property
    def job_id(self) -> str:
        raw = f"{self.mode}|{self.prompt}|{self.seed}|{self.width}x{self.height}"
        return hashlib.sha256(raw.encode()).hexdigest()[:12]

RUBRIC = {"prompt_adherence": 0.35, "aesthetics": 0.25, "text_rendering": 0.10, "anatomy": 0.15, "safety_pass": 0.15}

def score_sample(scores: dict[str, float]) -> float:
    return sum(scores[k] * w for k, w in RUBRIC.items())

job = ImageJob(prompt="aurora over fjord, cinematic wide shot", seed=7)
mock = {"prompt_adherence": 0.9, "aesthetics": 0.8, "text_rendering": 0.5, "anatomy": 0.7, "safety_pass": 1.0}
print(job.job_id, "weighted=", round(score_sample(mock), 3))


In [ ]:
# Demo 2: deterministic toy PNG generator (stdlib)
from pathlib import Path
import struct, zlib, hashlib

def write_minimal_png(path: str, w: int, h: int, rgb=(30, 60, 90)) -> str:
    def chunk(tag: bytes, data: bytes) -> bytes:
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data) & 0xffffffff)
    raw = b"".join(b"\x00" + bytes(rgb) * w for _ in range(h))
    ihdr = struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0)
    png = b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", ihdr) + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b"")
    Path(path).write_bytes(png)
    return path

def toy_generate(prompt: str, size: int = 64, path: str = "toy_gen.png") -> dict:
    h = hashlib.sha256(prompt.encode()).digest()
    rgb = (40 + h[0] % 180, 40 + h[1] % 180, 40 + h[2] % 180)
    write_minimal_png(path, size, size, rgb)
    return {"path": path, "rgb": rgb, "prompt_preview": prompt[:40]}

print(toy_generate("aurora over fjord"))
print(toy_generate("neon alley rain"))


## Training, Inference, and Alignment

### Definition
**Training** learns a generative distribution; **inference** samples under conditioning; **alignment** steers toward preference, brand, and safety.

### Why it matters
Skipping alignment ships pretty but risky images. Conflating stages leads to impossible product promises.

### How it works
Pretrain on image–text pairs → fine-tunes → preference/filters → inference with guidance, LoRAs, safety classifiers.

### Intuition
School → apprenticeship → company handbook → daily shift.

### Pitfalls
- Assuming quality is only 'more parameters'
- No eval suite after model swaps
- Safety only as a prompt instruction

### When to use
Design any production image feature with all three stages explicit.


### Stage comparison

| Stage | Goal | Artifacts |
|-------|------|-----------|
| Training | Learn p(image\|condition) | Checkpoints, VAEs, text encoders |
| Inference | Sample under latency/cost | Seeds, samplers, CFG, batching |
| Alignment | Prefer safe/useful/on-brand | Filters, reward models, C2PA |

```
[datasets] -> train -> [weights] -> align/filters
[prompt+controls] -> infer -> [pixels] -> [policy gate] -> user
```


In [ ]:
# Demo 3: OpenAI Images API shapes (placeholders)
import json
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
request_body = {
    "model": "gpt-image-1",
    "prompt": "Product photo of matte black headphones on marble, softbox lighting",
    "size": "1024x1024",
    "quality": "high",
    "n": 1,
}
response_body = {
    "created": 1710000000,
    "data": [{"b64_json": "<base64-png-bytes>", "revised_prompt": "Matte black over-ear headphones..."}],
}
print("POST /v1/images/generations")
print("Authorization: Bearer", OPENAI_API_KEY[:8] + "...")
print(json.dumps(request_body, indent=2))
print(json.dumps(response_body, indent=2)[:300], "...")


In [ ]:
# Demo 4: safety gate before returning bytes
DISALLOWED = {"weapon_build", "sexual_minor", "real_person_deepfake"}

def classify_risk(prompt: str) -> set[str]:
    p = prompt.lower(); hits = set()
    if "deepfake" in p or "celebrity face" in p: hits.add("real_person_deepfake")
    if "child" in p and "nude" in p: hits.add("sexual_minor")
    if "how to build a bomb" in p: hits.add("weapon_build")
    return hits

def gate(prompt: str) -> dict:
    blocked = sorted(classify_risk(prompt) & DISALLOWED)
    if blocked: return {"status": "blocked", "reasons": blocked}
    return {"status": "ok", "job": ImageJob(prompt=prompt).job_id}

for prompt in ["watercolor fox in snow", "celebrity face deepfake smiling", "child nude portrait"]:
    print(prompt[:32], "->", gate(prompt))


## Safety, Rights, and Provenance

### Definition
Safety covers disallowed content; rights cover training data, likeness, trademarks; provenance (C2PA) records how an asset was made.

### Why it matters
Legal and trust failures dominate headline risk.

### How it works
Layer prompt filters → model refusals → output classifiers → human review → watermark/credentials → audit logs.

### Intuition
Airport security + customs + boarding pass — multiple independent checks.

### Pitfalls
- Relying only on user ToS
- No likeness policy
- Logging prompts with PII forever

### When to use
Always for consumer and brand-facing generators.


### Rights & safety checklist

| Concern | Starter control |
|---------|-----------------|
| Training provenance | Licensed / disclosed datasets; document unknowns |
| Output filters | Multi-label classifiers + prefilters |
| Likeness / IP | Blocklist + similarity search |
| Watermarking | Invisible watermark + C2PA |
| Abuse | Rate limits, trust tiers, reporting |


### Try it yourself — Fundamentals

1. Extend ImageJob with strength for img2img (validate 0..1).
2. Add brand_fit and latency_sla rubric axes; renormalize weights.
3. Write 10 golden prompts and score a real API sample locally (YOUR_* keys).

**Stretch:** Design a C2PA-like metadata dict for every image.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `conditioning` | Extra signal steering generation |
| `latent space` | Compressed space where diffusion often runs |
| `CFG` | Classifier-free guidance strength |
| `seed` | RNG initializer for reproducible sampling |
| `C2PA` | Content Credentials provenance standard |
| `alignment` | Steering toward preferred/safe behavior |


### Workshop — Parameter journal — Image Fundamentals

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Fundamentals
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Fundamentals

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Fundamentals
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Fundamentals

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Fundamentals
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Fundamentals

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Fundamentals
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Fundamentals

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Fundamentals
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Fundamentals

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Fundamentals
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Fundamentals

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Fundamentals
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


## Key Takeaways

- Image gen = conditioning + sampling + safety/post layers
- Score adherence and safety, not vibes alone
- Separate training, inference, and alignment in architecture
- Log seeds, model IDs, and policy decisions for every asset
